In [ ]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [ ]:
from musicdb import MusicDBIO as PanDBIO
from gate import MusicDBGate
from match import MatchDBDataIO, AlbumReq
pdbio  = PanDBIO()
gate   = MusicDBGate()
mmeDF  = pdbio.getData()

# LastFM <=> MusicBrainz

In [ ]:
mdbio = gate.getIO("LastFM")
mbrainz = gate.getIO("MusicBrainz")

In [ ]:
searchData = mdbio.data.getSearchArtistData()
searchData['id'] = searchData['url'].apply(mdbio.getdbid)
searchData['MB'] = searchData['mbid'].apply(mbrainz.getdbid)

In [ ]:
from pandas import Series
mbidmapData = searchData[searchData["MB"].notna() & searchData["id"].notna()].drop_duplicates(subset=["mbid","MB"])
mbidMap = Series(mbidmapData['id'].values, index=mbidmapData["MB"].values)

In [ ]:
mmeDF["LastFMNew"] = mmeDF["MusicBrainz"].map(mbidmap)

In [ ]:
mmeDF = mmeDF.drop(["LastFM"], axis=1)
mmeDF = mmeDF.rename(columns={"LastFMNew": "LastFM"})

# SetListFM <=> MusicBrainz

In [ ]:
def mergeSetListFM(row, slfmMap):
    retval = row["SetListFM"] if isinstance(row["SetListFM"],str) else slfmMap.get(row["MusicBrainz"], None)
    return retval

mio = gate.getIO("SetListFM")
mbrainz = gate.getIO("MusicBrainz")
searchArtists      = mio.data.getSearchArtistData()
searchArtists['MB'] = searchArtists['mbid'].apply(mbrainz.getdbid)
slfmMap = searchArtists["id"]        
    
mmeDF["SetListFMNew"] = mmeDF[["MusicBrainz", "SetListFM"]].apply(mergeSetListFM, slfmMap=slfmMap, axis=1)

In [ ]:
mmeDF = mmeDF.drop(["SetListFM"], axis=1)
mmeDF = mmeDF.rename(columns={"SetListFMNew": "SetListFM"})

# Beatport From MusicBrainz

In [ ]:
from gate import MusicDBGate
gate  = MusicDBGate()
mdbio = gate.getIO("MusicBrainz")
bpio  = gate.getIO("Beatport")
bpMap = {}
mbMap = {}
for modVal in range(100):
    modValData = mdbio.data.getModValData(modVal)
    for mbid,mbidData in modValData.iteritems():
        if mbidData.profile.external is None:
            continue
        for item in mbidData.profile.external.get("Beatport", []):
            ref  = item.href if item is not None else None
            bpid = bpio.getdbid(ref) if isinstance(ref,str) else None
            if bpMap.get(mbid) is None:
                bpMap[mbid] = {}
            bpMap[mbid][(bpid,ref)] = mbidData.artist.name
    if modVal % 5 == 0:
        print(modVal,'\t',len(bpMap),'\t',len(mbMap))
bpMap = Series(bpMap)

In [ ]:
from pandas import Series,DataFrame
s = Series(bpMap)
beatportMap = {}
for k,v in bpMap[bpMap.map(len) == 1].iteritems():
    for k2,v2 in v.items():
        beatportMap[k] = k2[0]

for k,v in bpMap[bpMap.map(len) > 1].iteritems():
    for k2,v2 in v.items():
        print("beatportMap['{0}'] = {1: <10}  ## {2}  /  {3}".format(k,"'{0}'".format(k2[0]),k2[1],v2))
    print("")

In [ ]:
s = Series(beatportMap)
s.name = "Beatport"
df = DataFrame(s).join(mdbio.data.getSummaryNameData())

In [ ]:
externalsToGet = df[~df.index.isin(pdbio.getNotNaDBIDs("MusicBrainz")["MusicBrainz"])]
from ioutils import FileIO
io = FileIO()
io.save(idata=externalsToGet, ifile="beatportMap.p")

In [ ]:
for mbid,row in df[~df.index.isin(pdbio.getNotNaDBIDs("MusicBrainz")["MusicBrainz"])].iterrows():
    artistName = row["Name"]
    bpid = row["Beatport"]
    pdbio.newArtist(name=artistName, Beatport=bpid, MusicBrainz=mbid)

In [ ]:
toadd = externalsToGet.reset_index()
toadd = toadd.rename(columns={"index": "MusicBrainz", "Name": "ArtistName"})

In [ ]:
for i in range(toadd.shape[0]):

In [ ]:
from uuid import uuid4
idx = []
for i in range(toadd.shape[0]):
    idx.append(str(uuid4()))
toadd.index = idx

In [ ]:
from pandas import concat
pdbio.saveData(concat([mmeDF,toadd]))

In [ ]:
mmeDF["Beatport"] = mmeDF["MusicBrainz"].map(beatportMap)

In [ ]:
pdbio.saveData(mmeDF)

In [ ]:
mdioData

In [ ]:
baseDB    = "RateYourMusic"
baseDB    = "LastFM"
baseDB    = "MetalArchives"
baseReq   = {baseDB: AlbumReq(top=500)}

compareDBs  = ["RateYourMusic", "LastFM", "Spotify", "Genius", "Discogs", "MusicBrainz", "Deezer", "MetalArchives"]
compareReqs = {compareDB: AlbumReq(min=3) for compareDB in compareDBs if compareDB not in [baseDB]}
compareDBs  = list(compareReqs.keys())

albumReqs = {**baseReq, **compareReqs}
reqs       = {"Media": mediaTypes, "Albums": albumReqs, "Mask": baseDB, "NPart": 3, "Match": {"Artist": 0.8, "Medium": 2, "Tight": 1}}

print("Run Params:")
print("  ==> DBs:   [{0}] x {1}]".format(baseDB,list(compareReqs.keys())))
print("  ==> Media: {0}".format(mediaTypes))
print("  ==> Match: {0}".format(reqs["Match"]))